# Setup and Imports
This notebook loads the client deliverable artifacts and reproduces the core analysis views from the recorded experiment outputs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ks_2samp
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == '05_notebooks' else Path.cwd()
DATA_DIR = ROOT / 'deliverables' / '03_data'
GRAPH_DIR = ROOT / 'deliverables' / '02_graphs'

# Load Metrics Table
The full 72-run metrics table is the canonical summary of the experiment registry and is the starting point for all comparisons.

In [ ]:
metrics = pd.read_csv(DATA_DIR / 'all_metrics_72runs.csv')
metrics.head()

# Model Summary Statistics
Grouping by model makes the stability and average quality of each architecture easy to compare.

In [ ]:
summary = metrics.groupby('Model').agg({
    'Composite Score': ['mean', 'max'],
    'Physics Consistency': 'mean',
    'Statistical Similarity': 'mean',
    'ML Utility (TSTR)': 'mean',
    'Privacy / Detection AUC': 'mean'
}).round(4)
summary

# Reproduce Composite Score Ranking
This recreates the composite score ranking as a compact static plot directly from the parsed metrics file.

In [ ]:
order = metrics.groupby('Model')['Composite Score'].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
order.plot(kind='bar', color=['#00d4ff', '#4db06d', '#f59e0b', '#ef4444', '#8b5cf6', '#6b7280'])
plt.ylabel('Mean Composite Score')
plt.title('Composite Score Ranking')
plt.tight_layout()
plt.show()

# KDE Overlay for the Champion Model
We compare a few key numeric features between the original aerospace data and the best Empirical synthetic dataset.

In [ ]:
real = pd.read_csv(ROOT / 'aerospace_structural_design_dataset.csv')
synth = pd.read_csv(DATA_DIR / 'best_synthetic_empirical_seed11_n5000.csv')
features = ['E (GPa)', 'Tensile Strength (MPa)', 'ρ (kg/m³)', 'Altitude (m)']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.ravel(), features):
    real[feature].plot(kind='kde', ax=ax, label='Real', linestyle='--', color='black')
    synth[feature].plot(kind='kde', ax=ax, label='Synthetic', color='#00d4ff')
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()
plt.show()

# Simple TSTR Evaluation
A lightweight train-on-synthetic test-on-real check provides a transparent downstream utility sanity check.

In [ ]:
target = 'Weight Efficiency' if 'Weight Efficiency' in real.columns else 'Target — Weight Efficiency'
common = [c for c in real.columns if c in synth.columns]
real_common = real[common].copy()
synth_common = synth[common].copy()
X_syn = pd.get_dummies(synth_common.drop(columns=[target]))
X_real = pd.get_dummies(real_common.drop(columns=[target]))
X_syn, X_real = X_syn.align(X_real, join='outer', axis=1, fill_value=0)
y_syn = synth_common[target]
y_real = real_common[target]
clf = RandomForestClassifier(random_state=11)
clf.fit(X_syn, y_syn)
tstr = clf.score(X_real, y_real)
tstr

# KS Test Across Numeric Features
Feature-wise KS tests show how closely the champion synthetic dataset follows the empirical numeric marginals.

In [ ]:
numeric = [c for c in real.columns if c in synth.columns and pd.api.types.is_numeric_dtype(real[c])]
ks_df = pd.DataFrame([
    {'feature': col, 'p_value': ks_2samp(real[col], synth[col]).pvalue}
    for col in numeric
]).sort_values('p_value')
ks_df

# Correlation Matrix Comparison
This highlights whether the best synthetic dataset preserved the joint structure of the numeric engineering features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(real[numeric].corr(), vmin=-1, vmax=1, cmap='coolwarm')
axes[0].set_title('Real correlations')
axes[1].imshow(synth[numeric].corr(), vmin=-1, vmax=1, cmap='coolwarm')
axes[1].set_title('Synthetic correlations')
plt.tight_layout()
plt.show()

# Isolation Forest Novelty Detection
Training a novelty detector on the real data gives another quick view into whether the synthetic rows sit inside the real feature manifold.

In [ ]:
X_real_num = real[numeric]
X_syn_num = synth[numeric]
iso = IsolationForest(random_state=11)
iso.fit(X_real_num)
scores = iso.decision_function(X_syn_num)
pd.Series(scores).describe()

# Final Summary DataFrame
This final cell collects the most important headline metrics into one compact table for export or presentation.

In [ ]:
best_row = metrics.sort_values('Composite Score', ascending=False).iloc[0]
final = pd.DataFrame([
    {'metric': 'Champion model', 'value': best_row['Model']},
    {'metric': 'Champion seed', 'value': best_row['Seed']},
    {'metric': 'Champion size', 'value': best_row['Size']},
    {'metric': 'Champion composite', 'value': best_row['Composite Score']},
    {'metric': 'TSTR (simple notebook check)', 'value': tstr},
    {'metric': 'Mean novelty score', 'value': float(pd.Series(scores).mean())}
])
final